# Typewriter layout optimiser


## The model

The typewriter is treated as a single circular wheel of 38 positions (26 letters,
`,` `.`, and the ten digits). Space, newline and the other punctuation are handled by
separate mechanisms, so they are stripped from the corpus and the letters on either
side are treated as consecutive. 

$$\text{"100 years further maths"} \rarr \text{"100yearsfurthermaths"}$$

To type `b` right after `a` the wheel only turns **one way**, so the cost of the
bigram `a -> b` is the forward gap

$$\text{dist}(a \to b) = (\text{pos}[b] - \text{pos}[a]) \bmod 38$$

`layout_score` is the average number of $12\degree$ rotations per letter typed (lower = better). This is estimated for the english language based on this corpus of text https://raw.githubusercontent.com/O-X-E-Y/oxeylyzer/main/static/text/monkeyracer/mr.txt, and can be calculated by the equation below.

$$\text{score} = \frac{\sum_{(a,b)} \text{count}(a,b)\;\text{dist}(a \to b)}
                      {\sum_{(a,b)} \text{count}(a,b)}$$

In [30]:
import layout_optimiser as m


layouts = {
    "alphabetical": ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', ',', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9'],
    "QWERTY": ['q', 'w', 'e', 'r', 't', 'y', 'u', 'i', 'o', 'p', 'a', 's', 'd', 'f', 'g', 'h', 'j', 'k', 'l', 'z', 'x', 'c', 'v', 'b', 'n', 'm', ',', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9'],
    "Joe": ['t', 'h', 'e', 'r', 'a', 'i', 'o', 'n', 'd', 'g', 's', ',', '.', 'l', 'm', 'q', 'u', 'y', 'k', 'x', 'z', 'v', 'j', 'p', 'b', 'w', 'f', 'c', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9'],
}

score = lambda layout: m.layout_score(m.generate_layout_distances(layout), m.corpus_bigrams)
for name, layout in layouts.items():
    print(f"{name:<8} {score(layout):.4f}")

alphabetical 18.6891
QWERTY   18.6714
Joe      16.0562


## Finding an Optimal Arrangement

The `optimise` function is a greedy accent strategy where on each pass you simulate swapping every possible pair of keys and then see which one lowers the score the most. You then continuously do these passes until every possible swap would make the score worse, at which point, you have reached a local minimum.

To get as close to the optimal score as possible, and to know that the optimiser yields similar results no matter the initial configuration, `random_optimise` randomly shuffles the keys before running `optimise` and finding its local minimum. Below is an example of 10 locally optimal solutions.

In [28]:
SOLUTIONS = [
    ['t', 'z', 'w', 'h', 'b', 'v', 'e', 'a', 'y', 'j', 'o', 'q', 'u', 'r', 'l', 'i', 'n', 'x', 'd', 'g', '1', '9', '3', '7', '2', '6', '8', '5', '0', 's', '4', '.', ',', 'f', 'm', 'p', 'c', 'k'], 
    ['t', 'b', 'j', '7', 'o', '2', '4', '5', '0', 'f', 'q', 'u', 'k', 'm', 'p', 'w', 'h', 'v', 'e', 'x', 'a', 'r', 'l', 'i', 'n', 'd', 'g', '1', '9', '3', 'z', '6', 'y', 's', '8', '.', ',', 'c'],
    ['t', 'z', '9', '8', 'm', 'p', 'w', '3', '7', '2', '6', '4', '5', '0', 'h', 'b', 'v', 'e', 'x', 'a', 'l', 'i', 'n', 'd', 'g', 'y', 'c', 'j', 'o', 'f', 'q', 'u', 'r', 's', 'k', '1', '.', ','],
    ['t', 'z', '3', 'h', 'b', 'v', 'e', 'y', ',', 'j', 'o', 'x', 'f', 'q', 'u', '.', 'w', 'a', 'r', '2', '5', '0', 'l', 'i', 'n', 'd', 's', 'c', 'k', 'g', 'm', '1', '9', '7', '4', 'p', '6', '8'], 
    ['t', '1', '9', '2', '3', '4', '8', '5', 'w', 'h', 'i', 'z', '0', 'm', 'b', 'v', 'e', 'x', '7', 'a', 'n', 'l', 'd', 'g', 'y', 's', '6', ',', 'c', 'j', 'o', 'f', 'q', 'u', 'p', 'r', 'k', '.'], 
    ['t', '1', '9', 'b', 'j', 'o', 'f', 'q', 'u', 'z', '8', '3', '7', '2', '4', '5', '0', 'p', 'r', 'l', 'k', 'w', 'h', 'm', 'v', 'e', 'a', 'i', 'n', 'd', 'g', 'x', '6', 'y', 's', '.', ',', 'c'], 
    ['t', 'w', 'h', 'm', 'b', 'v', 'e', '.', 'p', 'a', 'l', 'y', '1', '9', '3', 'j', '7', '2', '6', '4', '8', 'o', 'f', 'q', '5', '0', 'u', 'z', 'x', 'r', 'i', 'n', 'd', 'g', 's', ',', 'c', 'k'], 
    ['t', '2', '4', 'z', '5', '0', 'w', 'h', 'j', 'o', 'k', 'f', 'm', 'v', 'p', 'b', 'e', 'x', 'a', 'q', 'u', 'r', 'l', 'i', 'n', 'd', 'g', 'y', 's', '1', '9', '8', '3', '7', '6', '.', ',', 'c'], 
    ['t', 'z', '4', 'w', 'h', 'm', 'v', 'p', 'e', 'x', '1', '9', '7', '2', '3', '6', '8', '5', '.', ',', '0', 'a', 'l', 'i', 'n', 'd', 'g', 'y', 'c', 'b', 'j', 'o', 'f', 'q', 'u', 'r', 's', 'k'], 
    ['t', 'w', 'h', 'i', 'm', 'v', 'e', 'a', 'n', 'd', 'g', '1', '9', 'x', '3', 'z', '7', '6', 'y', 's', 'c', 'b', 'q', 'j', 'o', 'u', 'p', 'r', '2', '4', 'l', '8', '5', 'k', ',', '0', '.', 'f']
]

scores = []
for i, layout in enumerate(SOLUTIONS, 1):
    s = score(layout)
    scores.append(s)
    print(f"Solution {i:>2}: {s:.4f}")

score_range = max(scores) - min(scores)
mean = sum(scores) / len(scores)

print()
print(f"Range: {score_range:.4f}  Mean: {mean:.3f}")

Solution  1: 14.8950
Solution  2: 14.8987
Solution  3: 15.0154
Solution  4: 14.9186
Solution  5: 15.1309
Solution  6: 15.0909
Solution  7: 14.9332
Solution  8: 14.8751
Solution  9: 15.0407
Solution 10: 15.2165

Range: 0.3414  Mean: 15.002


As you can see all of the local minimums are roughly 15 showing that is unlikely there exists a significantly better solution. 